In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
import openai, json

client = openai.OpenAI()

messages = [{
      "role": "system",
      "content": "너는 영화 전문가 에이전트다. 사용자 질문에 가장 알맞은 함수를 호출하고, 그 결과를 바탕으로 한국어로 친절하게 답해라."
  }]

In [19]:
import httpx

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    return httpx.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    return httpx.get(f"{BASE_URL}/movies/{id}").json()

def get_movie_credits(id):
    return httpx.get(f"{BASE_URL}/movies/{id}/credits").json()
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits
}



In [20]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 가져온다. 인자 없음",
            "parameters": {
                "type": "object", "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID를 받아서 영화 상세 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID, 예: 550"}
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 영화 ID를 받아서 영화 출연진 및 제작진 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID, 예: 550"},
                },
                "required": ["id"]
            }
        }
    }
]

In [21]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(message)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [22]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화가 무엇인지 알려줘
Calling function: get_popular_movies with {}
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession (2026-05-13 개봉)**
   - 장르: 공포, 스릴러
   - 평점: 7.9
   - 개요: "One Wish Willow"를 깨뜨린 한 남자가 자신의 욕망이 어두운 대가를 요구한다는 것을 알게 되는 이야기입니다.
   ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi (2026-06-03 개봉)**
   - 장르: 액션, 드라마
   - 평점: 5.9
   - 개요: 1980년대 안드라프라데시의 농촌에서 한 마을 사람이 스포츠로 지역사회의 단결을 이끌어내는 이야기입니다.
   ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Lee Cronin's The Mummy (2026-04-15 개봉)**
   - 장르: 공포, 미스터리
   - 평점: 8.1
   - 개요: 기자의 어린 딸이 사막에서 실종되고 8년 후 돌아오지만, 그 재회는 악몽으로 변해버리는 이야기입니다.
   ![Lee Cronin's The Mummy](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

4. **Kara (2026-04-30 개봉)**
   - 장르: 범죄, 드라마, 스릴러
   - 평점: 6.3
   - 개요: 범죄에서 벗어나려 하는 도둑이 아버지의 빚 문제로 인해 다시 범죄의 길로 돌아가는 이야기입니다.
   ![Kara](https://image.tmdb.org/t/p/w780/6U6i4qhgHR1MWkUb6OGQwNpqcZC.jpg)

5. **The Mandalorian and Gro